# Significance tests: seed-level paired t / Wilcoxon + per-user bootstrap CIs

Two complementary tests for SASRec vs. IA-SASRec variants:

1. **Seed-level paired t-test (+ Wilcoxon)** — unit of variation = random seed. Already implemented in `evaluation.paired_significance`. Same seed ⇒ same data split on both sides, so we use the *paired* t-test rather than Welch's (strictly more powerful when pairing is valid).
2. **Per-user paired bootstrap** — unit of variation = test user (thousands of them). Per-seed: score every test user with both checkpoints, take per-user metric diff, resample users with replacement, report 95% CI on the mean diff. Aggregated across seeds.

Reuses existing checkpoints under `results/checkpoints/` — file names live in each eval JSON's `checkpoint` field.

Per-user metrics are cached to parquet under `results/per_user/` so re-running is cheap.

In [ ]:
import sys, os, json, math
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import torch
from scipy import stats as sps

import config as cfg
from runner import _build_config, _instantiate_model, load_seed_result
from evaluation import paired_significance
from recbole.data import create_dataset, data_preparation

PROJECT = Path(cfg.PROJECT_ROOT)
EVAL_DIR = Path(cfg.EVAL_DIR)
CKPT_DIR = Path(cfg.CHECKPOINT_DIR)
PER_USER_DIR = PROJECT / 'results' / 'per_user'
PER_USER_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = cfg.DEVICE
print('device:', DEVICE)

## 1. Seed-level paired test (uses existing helper)

In [ ]:
DATASETS = ['ml-100k-iar', 'ml-1m', 'amazon-digital-music', 'amazon-office-products']
BASELINE = 'SASRec'
VARIANTS = ['IA-SASRec-Add', 'IA-SASRec-Mul', 'IA-SASRec-Val']
METRICS = ['ndcg@10', 'mrr@10', 'recall@10', 'ndcg@20', 'recall@20']

seed_rows = []
for ds in DATASETS:
    df = paired_significance(ds, BASELINE, VARIANTS, metrics=METRICS)
    if df.empty:
        continue
    df.insert(0, 'dataset', ds)
    seed_rows.append(df)
seed_df = pd.concat(seed_rows, ignore_index=True) if seed_rows else pd.DataFrame()

def _sig(p):
    if pd.isna(p): return ''
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    if p < 0.10: return '.'
    return ''

view = seed_df.copy()
view['t_sig'] = view['t_pvalue'].map(_sig)
view['w_sig'] = view['wilcoxon_pvalue'].map(_sig)
view[['dataset','model','metric','n','baseline_mean','challenger_mean','rel_diff_%','t_pvalue','t_sig','wilcoxon_pvalue','w_sig']]

## 2. Per-user metric computation

For each (dataset, model, seed) we:
1. Rebuild the RecBole `Config`/`Dataset`/dataloaders with the saved `best_params` and seed.
2. Load the checkpoint's `state_dict` into a fresh model.
3. For every test user: score all items, mask the user's history + padding, find the rank of the ground-truth target.
4. Derive per-user `recall@k`, `ndcg@k`, `mrr@k` (k ∈ {10, 20, 50, 100}).
5. Cache result to `results/per_user/<dataset>__<model>__seed<seed>.parquet`.

Note: this assumes leave-one-out (`LS: valid_and_test`) ⇒ exactly one ground-truth item per test user, which matches the project's eval config.

In [ ]:
def _per_user_path(dataset: str, model: str, seed: int) -> Path:
    return PER_USER_DIR / f'{dataset}__{model}__seed{seed}.parquet'

K_LIST = [10, 20, 50, 100]

def compute_per_user(dataset: str, model: str, seed: int, force: bool = False) -> pd.DataFrame:
    """Return per-user metrics DataFrame; cache to parquet."""
    out = _per_user_path(dataset, model, seed)
    if out.exists() and not force:
        return pd.read_parquet(out)

    rec = load_seed_result(dataset, model, seed)
    best_params = dict(rec.get('best_params') or {})
    ckpt_name = rec.get('checkpoint')
    if not ckpt_name:
        raise FileNotFoundError(f'no checkpoint recorded for {dataset}/{model}/seed={seed}')
    ckpt_path = CKPT_DIR / ckpt_name
    if not ckpt_path.exists():
        raise FileNotFoundError(f'missing checkpoint file: {ckpt_path}')

    overrides = dict(best_params)
    overrides['seed'] = seed
    rb_config, model_cls = _build_config(dataset, model, overrides, epochs=1, saved=False)
    rb_dataset = create_dataset(rb_config)
    _train, _valid, test_data = data_preparation(rb_config, rb_dataset)
    net = _instantiate_model(rb_config, test_data._dataset, model_cls)

    state = torch.load(ckpt_path, map_location=DEVICE)
    sd = state.get('state_dict', state)
    net.load_state_dict(sd)
    net.eval()

    uid_field = rb_dataset.uid_field
    iid_field = rb_dataset.iid_field

    rows = []
    with torch.no_grad():
        for batch in test_data:
            # FullSort dataloaders yield (interaction, history, positive_u, positive_i)
            if isinstance(batch, tuple) and len(batch) == 4:
                interaction, history, positive_u, positive_i = batch
            else:
                # Some sequential dataloaders return only the interaction.
                interaction = batch
                history = None
                positive_u = torch.arange(len(interaction[uid_field]))
                positive_i = interaction[iid_field]

            interaction = interaction.to(DEVICE)
            scores = net.full_sort_predict(interaction)
            n_users = interaction[uid_field].shape[0]
            if scores.dim() == 1:
                scores = scores.view(n_users, -1)
            scores[:, 0] = -float('inf')  # padding

            if history is not None:
                history_u, history_i = history
                if history_u.numel() > 0:
                    scores[history_u.to(DEVICE), history_i.to(DEVICE)] = -float('inf')

            uid_tokens = rb_dataset.id2token(uid_field, interaction[uid_field].cpu().numpy())

            # In leave-one-out + full eval, positive_u indexes into the batch's user rows,
            # and positive_i is the held-out target item id.
            pu = positive_u.cpu().numpy()
            pi = positive_i.cpu().numpy()

            # Group ground-truth items per user (usually 1 per user; handle >=1).
            for row_idx in range(n_users):
                mask = pu == row_idx
                if not mask.any():
                    continue
                gt_items = pi[mask].astype(np.int64)
                row_scores = scores[row_idx]
                # rank = number of items with strictly greater score, +1
                gt_scores = row_scores[torch.as_tensor(gt_items, device=DEVICE)]
                # Best of the ground-truth items (smallest rank)
                gt_best_score = gt_scores.max().item()
                rank = int((row_scores > gt_best_score).sum().item()) + 1
                rec_row = {'user_id': str(uid_tokens[row_idx]), 'rank': rank}
                for k in K_LIST:
                    hit = 1.0 if rank <= k else 0.0
                    rec_row[f'recall@{k}'] = hit  # single ground truth ⇒ recall == hit
                    rec_row[f'hit@{k}'] = hit
                    rec_row[f'ndcg@{k}'] = (1.0 / math.log2(rank + 1)) if rank <= k else 0.0
                    rec_row[f'mrr@{k}'] = (1.0 / rank) if rank <= k else 0.0
                rows.append(rec_row)

    df = pd.DataFrame(rows)
    df.to_parquet(out, index=False)
    return df

# Quick sanity check on a single seed.
_demo = compute_per_user('amazon-office-products', 'SASRec', 2020)
print('users:', len(_demo))
print('aggregate NDCG@10 (per-user):', _demo['ndcg@10'].mean())
print('aggregate NDCG@10 (eval JSON):', load_seed_result('amazon-office-products','SASRec',2020)['test_result']['ndcg@10'])

**Sanity check above:** the per-user mean of `ndcg@10` should match the aggregate `ndcg@10` from the eval JSON (within float noise). If not, the masking or ground-truth extraction is off and the rest of this notebook is invalid — stop and debug.

In [ ]:
# Compute per-user metrics for every (dataset, model, seed) that has a checkpoint.
TO_RUN = []
for ds in DATASETS:
    for m in [BASELINE] + VARIANTS:
        for f in sorted(EVAL_DIR.glob(f'{ds}__{m}__seed*.json')):
            seed = int(f.stem.rsplit('seed', 1)[1])
            try:
                rec = json.loads(f.read_text())
            except Exception:
                continue
            if not rec.get('checkpoint'):
                continue
            TO_RUN.append((ds, m, seed))

print(f'tasks: {len(TO_RUN)}')
for ds, m, s in TO_RUN:
    out = _per_user_path(ds, m, s)
    if out.exists():
        continue
    try:
        compute_per_user(ds, m, s)
        print(f'  ok   {ds}/{m}/seed={s}')
    except Exception as e:
        print(f'  FAIL {ds}/{m}/seed={s}: {e}')

## 3. Paired per-user bootstrap CIs + effect size

For each (dataset, variant, metric):

1. For every seed present in *both* baseline and variant, align users (inner-join on `user_id`) and compute per-user diff `d_u = variant(u) − baseline(u)`.
2. **Per-seed bootstrap (rigorous):** for each seed independently, resample users with replacement `B=2000` times, report the 95% CI on the mean diff. Then summarise across seeds: how many seeds have CI entirely > 0? entirely < 0?
3. **Pooled bootstrap (coarse summary):** concatenate diffs across seeds and bootstrap once. *Caveat:* this treats `(user_u, seed_2020)` and `(user_u, seed_2021)` as exchangeable, which they aren't — same user implies correlated errors. Use per-seed CIs as the rigorous reading; the pooled CI is for at-a-glance comparison only.
4. **Cohen's d:** `mean(d_u) / std(d_u)`, per seed and pooled. With thousands of test users a p-value becomes trivially small for tiny effects — d quantifies whether the gain is *practically* meaningful. Rules of thumb: |d| ≥ 0.2 small, ≥ 0.5 medium, ≥ 0.8 large. For top-k rec-sys gains, expect tiny d (often 0.02–0.05) even when p ≪ 0.05; that's the practical-significance reality check.
5. **Two-sided bootstrap p-value:** `2 * min(P(mean<=0), P(mean>=0))`, floored at `1/B`.

Note on the seed-variance question: bootstrap-resampling users does not account for training instability across seeds. With n=3–5 seeds, a nested "resample seeds then users" bootstrap is degenerate (the outer resample has too few distinct multisets). The honest treatment is to report the seed-level paired test (§1) *alongside* the per-user bootstrap — they answer different questions and a finding worth publishing should survive both.

In [ ]:
def _bootstrap_ci(diffs: np.ndarray, n_boot: int, rng: np.random.Generator) -> tuple[float, float, float]:
    """Return (ci_lo, ci_hi, two_sided_p) from a paired-diff bootstrap.

    Uses percentile CI and the proportion-of-resamples definition of p,
    floored at 1/n_boot to avoid p=0.
    """
    n = len(diffs)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = diffs[idx].mean(axis=1)
    lo, hi = np.quantile(boot_means, [0.025, 0.975])
    n_le = int((boot_means <= 0).sum())
    n_ge = int((boot_means >= 0).sum())
    p = 2.0 * min(n_le, n_ge) / n_boot
    return float(lo), float(hi), max(p, 1.0 / n_boot)


def _cohens_d(diffs: np.ndarray) -> float:
    """Cohen's d on paired diffs: mean / std (unbiased)."""
    if len(diffs) < 2:
        return float('nan')
    s = diffs.std(ddof=1)
    return float(diffs.mean() / s) if s > 0 else float('nan')


def paired_user_bootstrap(
    dataset: str,
    variant: str,
    metric: str,
    n_boot: int = 2000,
    rng: np.random.Generator | None = None,
) -> dict:
    rng = rng or np.random.default_rng(0)
    base_files = sorted(PER_USER_DIR.glob(f'{dataset}__{BASELINE}__seed*.parquet'))
    base_seeds = {int(p.stem.rsplit('seed', 1)[1]): p for p in base_files}
    var_files = sorted(PER_USER_DIR.glob(f'{dataset}__{variant}__seed*.parquet'))
    var_seeds = {int(p.stem.rsplit('seed', 1)[1]): p for p in var_files}
    shared = sorted(set(base_seeds) & set(var_seeds))
    if not shared:
        return {}

    # Per-seed: bootstrap independently, Cohen's d, store.
    per_seed = []
    pooled_diffs = []
    pooled_base_means = []
    for s in shared:
        b = pd.read_parquet(base_seeds[s], columns=['user_id', metric]).rename(columns={metric: 'b'})
        c = pd.read_parquet(var_seeds[s], columns=['user_id', metric]).rename(columns={metric: 'c'})
        m = b.merge(c, on='user_id', how='inner')
        d = (m['c'] - m['b']).to_numpy()
        pooled_diffs.append(d)
        pooled_base_means.append(m['b'].mean())
        lo, hi, p = _bootstrap_ci(d, n_boot, rng)
        per_seed.append({
            'seed': s,
            'n_users': len(d),
            'mean_diff': float(d.mean()),
            'ci_lo': lo,
            'ci_hi': hi,
            'p_boot': p,
            'cohens_d': _cohens_d(d),
            'sig_pos': lo > 0,
            'sig_neg': hi < 0,
        })

    diffs_pool = np.concatenate(pooled_diffs)
    base_mean = float(np.mean(pooled_base_means))
    lo_p, hi_p, p_pool = _bootstrap_ci(diffs_pool, n_boot, rng)
    obs_mean = float(diffs_pool.mean())
    n_seeds_sig_pos = sum(1 for r in per_seed if r['sig_pos'])
    n_seeds_sig_neg = sum(1 for r in per_seed if r['sig_neg'])

    return {
        'dataset': dataset,
        'model': variant,
        'metric': metric,
        'n_seeds': len(shared),
        'n_users_total': len(diffs_pool),
        'baseline_mean': base_mean,
        'mean_diff': obs_mean,
        'rel_diff_%': (obs_mean / base_mean * 100.0) if base_mean else 0.0,
        'ci_lo_pool': lo_p,
        'ci_hi_pool': hi_p,
        'p_boot_pool': p_pool,
        'cohens_d_pool': _cohens_d(diffs_pool),
        'seeds_sig_pos': n_seeds_sig_pos,
        'seeds_sig_neg': n_seeds_sig_neg,
        'per_seed': per_seed,
    }


boot_rows = []
for ds in DATASETS:
    for v in VARIANTS:
        for met in METRICS:
            r = paired_user_bootstrap(ds, v, met)
            if r:
                boot_rows.append(r)
boot_df = pd.DataFrame(boot_rows)
boot_df['sig_pool'] = boot_df['p_boot_pool'].map(_sig)
boot_df['seeds_sig'] = boot_df.apply(lambda r: f"+{int(r['seeds_sig_pos'])}/-{int(r['seeds_sig_neg'])}/{int(r['n_seeds'])}", axis=1)
boot_df[['dataset','model','metric','n_seeds','n_users_total','baseline_mean','mean_diff','rel_diff_%','ci_lo_pool','ci_hi_pool','p_boot_pool','sig_pool','cohens_d_pool','seeds_sig']]

## 4. Combined view

In [ ]:
left = seed_df[['dataset','model','metric','n','rel_diff_%','t_pvalue','wilcoxon_pvalue']].rename(
    columns={'n':'n_seeds','rel_diff_%':'rel_diff_seed_%'}
)
right = boot_df[[
    'dataset','model','metric','n_users_total','rel_diff_%',
    'ci_lo_pool','ci_hi_pool','p_boot_pool','cohens_d_pool',
    'seeds_sig_pos','seeds_sig_neg',
]].rename(columns={'rel_diff_%':'rel_diff_user_%'})
combined = left.merge(right, on=['dataset','model','metric'], how='outer')
combined['t_sig'] = combined['t_pvalue'].map(_sig)
combined['w_sig'] = combined['wilcoxon_pvalue'].map(_sig)
combined['boot_sig'] = combined['p_boot_pool'].map(_sig)
combined['seeds_sig'] = combined.apply(
    lambda r: f"+{int(r['seeds_sig_pos'])}/-{int(r['seeds_sig_neg'])}/{int(r['n_seeds'])}"
    if pd.notna(r.get('seeds_sig_pos')) else '', axis=1
)
combined = combined.sort_values(['dataset','model','metric']).reset_index(drop=True)
combined

In [ ]:
# "Publishable" rows: seed-level paired test rejects AND majority of per-seed user-bootstrap CIs agree on sign
# AND pooled bootstrap rejects. The seeds_sig majority guards against one lucky seed driving the pool.
def _seed_majority(r):
    if pd.isna(r.get('n_seeds')):
        return False
    n = int(r['n_seeds'])
    return int(r['seeds_sig_pos']) > n/2 or int(r['seeds_sig_neg']) > n/2

robust = combined[
    ((combined['t_pvalue'] < 0.05) | (combined['wilcoxon_pvalue'] < 0.05))
    & (combined['p_boot_pool'] < 0.05)
    & combined.apply(_seed_majority, axis=1)
].copy()
robust[[
    'dataset','model','metric',
    'rel_diff_seed_%','rel_diff_user_%',
    't_pvalue','wilcoxon_pvalue',
    'ci_lo_pool','ci_hi_pool','p_boot_pool',
    'cohens_d_pool','seeds_sig',
]]